# Casecast · exploratory analysis

Real CDC weekly reported COVID-19 cases in California. This executed notebook inspects the cached snapshot and completed experiment. Modeling logic lives in `src/`; use `disease-forecast run` to reproduce forecasts. No synthetic data or clinical predictions are used.

In [1]:
import json
from pathlib import Path

import pandas as pd

from disease_forecasting.data import load_data

root = Path.cwd()
if not (root / "configs").exists():
    root = root.parent
data = load_data(root)
print(f"Weeks: {len(data)}; start: {data.index.min().date()}; end: {data.index.max().date()}")
print(data.head().to_string())

Weeks: 173; start: 2020-01-22; end: 2023-05-10
            cases
date             
2020-01-22      0
2020-01-29      2
2020-02-05      6
2020-02-12      8
2020-02-19     14


## Integrity and reporting distribution

Missing reports must not become zeros. The validator rejects missing or duplicate weeks, negative counts, non-integer counts, unexpected geography, and incomplete periods. The large peaks below are retained. Changes in reported cases can reflect both transmission and changes in testing/reporting.

In [2]:
print("Missing values:", int(data.cases.isna().sum()))
print("Source-reported zero weeks:", int(data.cases.eq(0).sum()))
print("Peak week:", data.cases.idxmax().date(), "cases:", int(data.cases.max()))
print(
    data.groupby(data.index.year).cases.agg(["count", "mean", "median", "max"]).round(1).to_string()
)

Missing values: 0
Source-reported zero weeks: 1
Peak week: 2022-01-12 cases: 790954
      count      mean   median     max
date                                  
2020     50   46040.7  23834.0  305346
2021     52   59809.8  40681.5  311715
2022     52  123091.8  68356.0  790954
2023     19   23100.3  23037.0   57501


## Seasonality is a hypothesis, not a guarantee

Lag correlations below summarize the entire descriptive series, not a model-selection procedure. Only about three annual cycles are available; COVID waves and reporting changes do not imply a stable annual seasonal pattern.

In [3]:
for lag in [1, 2, 4, 12, 52]:
    print(f"Lag {lag:2d} weeks: correlation {data.cases.autocorr(lag):.3f}")

Lag  1 weeks: correlation 0.912
Lag  2 weeks: correlation 0.717
Lag  4 weeks: correlation 0.286
Lag 12 weeks: correlation -0.133
Lag 52 weeks: correlation 0.202


## Honest held-out comparison

Model choice uses validation only. Calibration errors supply interval radii; final test errors are never used to tune parameters or intervals. All candidates share origins and horizons. This is a retrospective revised-data evaluation, not a reconstruction of real-time data vintages.

In [4]:
metadata = json.loads((root / "reports/run_metadata.json").read_text())
scores = pd.read_csv(root / "reports/test_metrics.csv")
print("Validation-selected model:", metadata["selected_model"])
print(
    scores[["model", "horizon", "n", "mae", "rmse", "coverage_95", "width_95"]]
    .round(2)
    .to_string(index=False)
)

Validation-selected model: Naive
         model  horizon  n       mae      rmse  coverage_95  width_95
    Damped ETS        1 23   7680.08  13216.25         0.96  59005.74
    Damped ETS        2 23  10505.19  16275.44         0.96  75373.22
    Damped ETS        4 23  19465.26  35968.15         0.96 137202.52
         Naive        1 23   7669.52  12282.48         1.00 132774.65
         Naive        2 23  10291.57  14621.08         1.00 171649.65
         Naive        4 23  17076.91  24126.97         1.00 199185.65
 Random Forest        1 23  13961.36  26592.53         1.00 288611.88
 Random Forest        2 23  47043.03  86069.54         0.87 209616.53
 Random Forest        4 23 130140.50 216230.60         0.70 178622.07
Seasonal naive        1 23 155203.22 289820.75         0.70 192990.83
Seasonal naive        2 23 156448.04 289892.16         0.70 193289.78
Seasonal naive        4 23 159464.00 290174.55         0.70 195100.65


## Interpretation

The naive baseline has the lowest test MAE at every horizon. More complex models did not improve generalization here. High interval coverage must be read alongside wide intervals and small, dependent test samples. Random Forest's longer-horizon errors show how unstable lag relationships can be across epidemic regimes. See [interactive historical chart](../reports/figures/historical_cases.html) and [dashboard screenshot](../reports/figures/dashboard.png).

**Not a diagnosis:** aggregate state case counts contain no individual symptoms, examinations, or test results. They cannot establish whether any person has a disease.